In [1]:
import sys
sys.path.insert(0, '..')

from helper.golbal_variables import * 
from helper.DataLoaders.for_normal_learning.wave2vec2_dataloader import Audio_DataLoader, collate_fn
from wav2vec2 import Wav2Vec2ForGenre

z:\dlgen-AI-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
z:\dlgen-AI-project\.venv\Lib\site-packages\transformers\configuration_utils.py:364: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


In [2]:
import torch 

device = "cuda" if torch.cuda.is_available() else "cpu"

num_classes = 10   # genres
model = Wav2Vec2ForGenre(num_classes).to(device)

z:\dlgen-AI-project\.venv\Lib\site-packages\transformers\configuration_utils.py:364: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
# from torch.utils.data import WeightedRandomSampler, DataLoader, Subset
# from collections import Counter
# import numpy as np

# dataset = Audio_DataLoader(
#     genre_dir=GENRES_STEMS,
#     esc50_dir=ESC_50_MASTER
# )
# num_songs = len(dataset.samples)
# song_indices = np.arange(num_songs)
# np.random.seed(42)
# np.random.shuffle(song_indices)

# train_song_count = int(0.8 * num_songs)
# train_songs = set(song_indices[:train_song_count])
# val_songs = set(song_indices[train_song_count:])

# print(f"Songs split: {len(train_songs)} train, {len(val_songs)} val")

# train_indices = [i for i, (song_idx, start) in enumerate(dataset.index_map) 
#                  if song_idx in train_songs]
# val_indices = [i for i, (song_idx, start) in enumerate(dataset.index_map) 
#                if song_idx in val_songs]

# # Create subsets
# train_dataset = Subset(dataset, train_indices)
# val_dataset = Subset(dataset, val_indices)

# print(f"Train chunks: {len(train_dataset)}")
# print(f"Val chunks: {len(val_dataset)}")


# # Get labels for training chunks
# train_labels = [dataset.labels[dataset.index_map[i][0]] for i in train_indices]

# class_counts = Counter(train_labels)

# class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
# sample_weights = [class_weights[label] for label in train_labels]

# sampler = WeightedRandomSampler(
#     weights=sample_weights,
#     num_samples=len(sample_weights),
#     replacement=True
# )

# print("\nClass distribution in training chunks:")
# genre_names = {v: k for k, v in dataset.class_map.items()}
# for cls, count in sorted(class_counts.items()):
#     print(f"  {genre_names[cls]}: {count} chunks (weight: {class_weights[cls]:.3f})")

# train_dataloader = DataLoader(
#     dataset=train_dataset,
#     batch_size=16,
#     sampler=sampler,
#     collate_fn=collate_fn,
#     num_workers=0,
#     pin_memory=True if device == "cuda" else False
# )

# val_dataloader = DataLoader(
#     dataset=val_dataset,
#     batch_size=16,
#     shuffle=False,
#     collate_fn=collate_fn,
#     num_workers=0,
#     pin_memory=True if device == "cuda" else False
# )

# print(f"\n{'='*70}")
# print(f"DATALOADERS READY")
# print(f"{'='*70}")
# print(f"Train batches: {len(train_dataloader)}")
# print(f"Val batches: {len(val_dataloader)}")
# print(f"{'='*70}\n")

In [4]:
# from utils import train_wav2vec2_model 

# train_wav2vec2_model(
#     model=model,
#     device=device,
#     train_loader=train_dataloader,
#     val_loader=val_dataloader,
#     epochs=30
# )

In [5]:
import sys
sys.path.insert(0, '..')

import torch 

from helper.golbal_variables import * 
from wav2vec2 import Wav2Vec2ForGenre

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
model = Wav2Vec2ForGenre(num_classes=10)
model.load_state_dict(torch.load("best_wav2vec2_genre.pth", map_location=device))
model = model.to(device)

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\visha\AppData\Local\Temp\ipykernel_26996\3527579559.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed 

In [8]:
from predict import predict 
import pandas as pd

genre_map = {
    'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4,
    'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9
}

In [9]:
test_df = pd.read_csv(MESSY_DATA_DIR / "test.csv")
output = {}
for idx, row in test_df.iterrows():
    song_id = row["id"]
    filename = row["filename"]
    audio_path = MESSY_DATA_DIR / filename

    predicted_genre, confidence, all_probs = predict(
        model=model,
        audio_path=audio_path,
        device=device,
        genre_map=genre_map
    )

    output[song_id] = predicted_genre
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(test_df)} songs...")

output_df = pd.DataFrame(list(output.items()), columns=['id', 'genre'])
output_df.set_index("id", inplace=True)
output_df.to_csv("output.csv")

Processed 100/3020 songs...
Processed 200/3020 songs...
Processed 300/3020 songs...
Processed 400/3020 songs...
Processed 500/3020 songs...
Processed 600/3020 songs...
Processed 700/3020 songs...
Processed 800/3020 songs...
Processed 900/3020 songs...
Processed 1000/3020 songs...
Processed 1100/3020 songs...
Processed 1200/3020 songs...
Processed 1300/3020 songs...
Processed 1400/3020 songs...
Processed 1500/3020 songs...
Processed 1600/3020 songs...
Processed 1700/3020 songs...
Processed 1800/3020 songs...
Processed 1900/3020 songs...
Processed 2000/3020 songs...
Processed 2100/3020 songs...
Processed 2200/3020 songs...
Processed 2300/3020 songs...
Processed 2400/3020 songs...
Processed 2500/3020 songs...
Processed 2600/3020 songs...
Processed 2700/3020 songs...
Processed 2800/3020 songs...
Processed 2900/3020 songs...
Processed 3000/3020 songs...


In [10]:
output_df["genre"].unique()

<ArrowStringArray>
[      'pop', 'classical',     'disco',    'hiphop',   'country',      'rock',
     'metal',      'jazz',    'reggae',     'blues']
Length: 10, dtype: str